# PubMed Oncologist — DPO Training Data Generator

Generate **DPO preference pairs** for Direct Preference Optimization training.

**Prerequisites:** Run `pubmed_datagen_sft.ipynb` first — this notebook reads the SFT artifacts
(grounding rejects, beyond-evidence QA, self-correction sequences) to build DPO pairs.

**Philosophy:** The model should behave like a well-read oncologist — freely using established
medical knowledge while never fabricating specific data. DPO "chosen" responses draw on broad
knowledge; "rejected" responses invent fake statistics, trial names, or study details.

**Three DPO pair sources:**

| Source | Chosen (Good) | Rejected (Bad) | Signal |
|--------|--------------|----------------|--------|
| **Grounding Rejects** | Knowledgeable answer (abstract + general knowledge) | Fabricated-data answer (from grounding check) | Don't invent specifics |
| **Beyond-Evidence** | Knowledgeable answer acknowledging evidence gaps | Fabricated-data answer (newly generated) | Be transparent about evidence strength |
| **Self-Correction** | Corrected answer | Flawed answer | Fix mistakes when caught |

**Output:** `pubmed_oncologist_v2_dpo.jsonl` — TRL chat-template DPO pairs

**Next step:** Run `pubmed_dpo_training_v2.ipynb` to train the DPO LoRA.

## 1. Configuration

In [ ]:
!pip install dotenv
import os
from dotenv import load_dotenv

# Load .env file (works in Docker, JupyterLab, and local environments)
load_dotenv()

# v2 datagen config
# =========================== API CONFIGURATION ===========================
# Qwen3-235B THINKING model — enforces <think> blocks for clinical reasoning
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not set.\n"
        "Options:\n"
        "  1. Create a .env file in the project root with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Export in your shell:  export OPENROUTER_API_KEY=sk-or-...\n"
        "  3. Pass via Docker:      docker run -e OPENROUTER_API_KEY=sk-or-... ..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-thinking-2507"
MODEL_LITE = "qwen/qwen-2.5-7b-instruct"       # cheaper model for classification & question generation

# =========================== THINKING CONFIGURATION =======================
# KEEP_THINKING: Preserve <think>...</think> blocks in training data.
# This teaches the LoRA to reason step-by-step before answering.
# Set False ONLY if you want answer-only training (not recommended for medical).
KEEP_THINKING = True

# =========================== GENERATION PARAMETERS ========================
CONCURRENCY = 50
TEMPERATURE_ANSWERS = 0.4
TEMPERATURE_QUESTIONS = 0.8
NUM_ROUNDS = 1
QUESTIONS_PER_CHUNK = 3
TURNS_PER_CONVERSATION = 4

# =========================== TEST MODE ====================================
TEST_CHUNKS_PER_ROUND = 5000   # Set to 0 for full run

# =========================== CHUNKING PARAMETERS =========================
CHUNK_SIZE = 1500       # characters per chunk
OVERLAP_SENTENCES = 2   # pySBD sentence overlap

# =========================== SAMPLING =====================================
MAX_RECORDS = 2    # max total records to use (proportional random sampling)

# =========================== CANCER TYPES =================================
# Maps source-clean filenames to display labels
CANCER_TYPES = {
    "pubmed_bone_cancer": "Bone Cancer",
    "pubmed_brain_tumour": "Brain Tumour",
    "pubmed_breast_cancer": "Breast Cancer",
    "pubmed_colon_cancer": "Colon Cancer",
    "pubmed_gastric_cancer": "Gastric Cancer",
    "pubmed_kidney_cancer": "Kidney Cancer",
    "pubmed_lung_cancer": "Lung Cancer",
    "pubmed_ovarian_cancer": "Ovarian Cancer",
    "pubmed_prostate_cancer": "Prostate Cancer",
    "pubmed_skin_cancer": "Skin Cancer",
}

# =========================== PATHS (auto-detect Docker vs host) ===========
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (JupyterLab container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR      = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
OUTPUT_ROOT   = f"{DATA_DIR}/training-data"
OUTPUT_DIR    = f"{OUTPUT_ROOT}/pubmed_oncologist_v2"
OUTPUT_FILE   = f"{OUTPUT_ROOT}/pubmed_oncologist_v2.jsonl"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("v2 Datagen Configuration")
print("=" * 50)
print(f"  Environment:   {_env}")
print(f"  PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"  Model (main): {MODEL_NAME} (thinking answers)")
print(f"  Model (lite): {MODEL_LITE} (classification & Q-gen)")
print(f"  Keep thinking: {KEEP_THINKING}")
print(f"  Concurrency:   {CONCURRENCY}")
print(f"  Temp answers:  {TEMPERATURE_ANSWERS}")
print(f"  Temp questions:{TEMPERATURE_QUESTIONS}")
print(f"  Rounds:        {NUM_ROUNDS}")
print(f"  Q per chunk:   {QUESTIONS_PER_CHUNK}")
print(f"  Chunk size:    {CHUNK_SIZE} chars, {OVERLAP_SENTENCES} sentence overlap")
print(f"  Max records:   {MAX_RECORDS:,}")
print(f"  Test chunks:   {TEST_CHUNKS_PER_ROUND} (0 = full run)")
print(f"  Cancer types:  {len(CANCER_TYPES)}")
print(f"  Output dir:    {OUTPUT_DIR}")
print(f"  Output file:   {OUTPUT_FILE}")

v2 Datagen Configuration
  Environment:   Docker (JupyterLab container)
  PROJECT_ROOT:  /workspace/training/pubmed
  Model (main): qwen/qwen3-235b-a22b-thinking-2507 (thinking answers)
  Model (lite): qwen/qwen-2.5-7b-instruct (classification & Q-gen)
  Keep thinking: True
  Concurrency:   50
  Temp answers:  0.4
  Temp questions:0.8
  Rounds:        1
  Q per chunk:   3
  Chunk size:    1500 chars, 2 sentence overlap
  Max records:   2
  Test chunks:   0 (0 = full run)
  Cancer types:  10
  Output dir:    /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2
  Output file:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2.jsonl


## 2. Environment

In [2]:
# Install deps, define ALL shared helpers used by every downstream cell.
# v2 fixes: _count_lines, grounding paths, conversations extraction.
# _count_lines, is_low_quality_answer, process_answer, has_thinking_block,
# _api_call_with_timeout, _extract_with_reasoning, semaphore, ApiErrorTracker,
# AsyncOpenAI client.  Also validates model IDs at startup (fail-fast).
%pip install openai tqdm nest_asyncio tiktoken pysbd -q

import asyncio
import json
import glob
import re
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter
from openai import AsyncOpenAI, OpenAI as SyncOpenAI
from tqdm.notebook import tqdm
from tqdm.asyncio import tqdm as atqdm
import nest_asyncio
import pysbd
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================================
# ANSWER QUALITY CHECK — reject low-quality or non-reasoning responses
# ============================================================================

def has_thinking_block(answer: str) -> bool:
    """Check if answer contains a <think>...</think> block."""
    return bool(re.search(r'<think>.*?</think>', answer, re.DOTALL))


def is_low_quality_answer(answer: str) -> bool:
    """Return True if the answer fails quality checks for medical CoT training."""
    if not answer or len(answer.strip()) < 50:
        return True
    lower = answer.strip().lower()
    if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but i"]):
        return True
    return False


def process_answer(answer: str, keep_thinking: bool = KEEP_THINKING) -> str:
    """Process a raw answer from the thinking model.

    If keep_thinking=True: preserve <think> blocks in training data.
    If keep_thinking=False: strip <think> blocks, keep only final answer.
    """
    if not answer:
        return ""

    if keep_thinking:
        return answer.strip()
    else:
        cleaned = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
        cleaned = re.sub(r'<think>.*', '', cleaned, flags=re.DOTALL).strip()
        cleaned = re.sub(r'</think>.*', '', cleaned, flags=re.DOTALL).strip()
        return cleaned


# ============================================================================
# GENERATION HELPERS
# ============================================================================

semaphore = asyncio.Semaphore(CONCURRENCY)


async def _api_call_with_timeout(coro, timeout_secs=180):
    """Wrap an API coroutine with a timeout.
    Thinking model needs longer timeout — reasoning takes time."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None


def _extract_with_reasoning(resp) -> str:
    """Extract answer content with reasoning from OpenRouter response.

    OpenRouter thinking models return reasoning in a separate 'reasoning' field
    (via model_extra), NOT inline <think> tags in content. This helper recombines
    them into the <think>...</think> format expected by process_answer().
    """
    msg = resp.choices[0].message
    content = (msg.content or "").strip()

    reasoning = ""
    if hasattr(msg, 'reasoning') and msg.reasoning:
        reasoning = msg.reasoning.strip()
    elif hasattr(msg, 'model_extra') and msg.model_extra and 'reasoning' in msg.model_extra:
        reasoning = (msg.model_extra['reasoning'] or "").strip()

    if reasoning:
        return f"<think>\n{reasoning}\n</think>\n\n{content}"
    return content


# ── Shared Error Tracker (used by ALL generation cells) ──────────────

def _count_lines(filepath: str) -> int:
    """Count non-empty lines in a file."""
    with open(filepath) as f:
        return sum(1 for line in f if line.strip())


class ApiErrorTracker:
    """Track API call outcomes for a generation phase. Fail-fast on high error rates."""
    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.errors = 0
        self.timeouts = 0
        self._samples = []

    def success(self):
        self.calls += 1

    def error(self, e: Exception):
        self.calls += 1
        self.errors += 1
        if len(self._samples) < 5:
            self._samples.append(f"{type(e).__name__}: {str(e)[:300]}")

    def timeout(self):
        self.calls += 1
        self.timeouts += 1

    @property
    def failed(self) -> int:
        return self.errors + self.timeouts

    def check_fatal(self, threshold: float = 0.95):
        """Raise RuntimeError if failure rate >= threshold."""
        if self.calls == 0:
            return
        rate = self.failed / self.calls
        if rate >= threshold:
            sample_str = '\n    '.join(self._samples) if self._samples else '(no details captured)'
            raise RuntimeError(
                f"\n{'='*60}\n"
                f"FATAL: {self.name} — {self.failed}/{self.calls} calls failed ({rate:.0%})\n"
                f"  Errors: {self.errors}, Timeouts: {self.timeouts}\n"
                f"  Sample errors:\n    {sample_str}\n"
                f"{'='*60}"
            )

    def report(self) -> str:
        """Return warning string if any failures, else empty string."""
        if self.failed == 0:
            return ""
        return (f"  ⚠ {self.name}: {self.errors} errors + {self.timeouts} timeouts "
                f"/ {self.calls} calls")


# ── AsyncOpenAI client (retries 429/5xx with backoff) ──
client = AsyncOpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    max_retries=3,
    timeout=180.0,
)

# ── Validate model IDs at startup (fail-fast on typos) ──
print("Validating model IDs on OpenRouter...")
_sync = SyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, timeout=30)
for _model_id in [MODEL_NAME, MODEL_LITE]:
    try:
        _sync.chat.completions.create(
            model=_model_id,
            messages=[{"role": "user", "content": "1+1="}],
            max_tokens=1,
        )
        print(f"  ✓ {_model_id}")
    except Exception as _e:
        raise ValueError(
            f"\n✗ INVALID MODEL ID: '{_model_id}'\n"
            f"  Error: {_e}\n"
            f"  Fix MODEL_NAME or MODEL_LITE in the config cell and re-run."
        ) from None
del _sync

print(f"\n✓ Environment ready — models validated")
print(f"  Main:  {MODEL_NAME}")
print(f"  Lite:  {MODEL_LITE}")
print(f"  Helpers: semaphore, _api_call_with_timeout, _extract_with_reasoning,")
print(f"           is_low_quality_answer, process_answer, has_thinking_block,")
print(f"           ApiErrorTracker — all defined")

Note: you may need to restart the kernel to use updated packages.
Validating model IDs on OpenRouter...
  ✓ qwen/qwen3-235b-a22b-thinking-2507
  ✓ qwen/qwen-2.5-7b-instruct

✓ Environment ready — models validated
  Main:  qwen/qwen3-235b-a22b-thinking-2507
  Lite:  qwen/qwen-2.5-7b-instruct
  Helpers: semaphore, _api_call_with_timeout, _extract_with_reasoning,
           is_low_quality_answer, process_answer, has_thinking_block,
           ApiErrorTracker — all defined


## 3. Load & Prepare Source Data

Loads pre-cleaned JSONL files from `source-clean/` (produced by `scripts/clean_pubmed.py`).

**Sentence-aware chunking** (pySBD + medical abbreviation protection):
- Splits text into sentences first using rules-based boundary detection
- Protects medical abbreviations (et al., i.v., vs., mos., pts., etc.) from false splits
- Groups complete sentences into chunks up to `CHUNK_SIZE`
- Overlap = last `OVERLAP_SENTENCES` trailing sentences from previous chunk (always complete)

In [3]:
# ============================================================================
# MEDICAL-AWARE SENTENCE TOKENIZER
# pySBD rules engine + abbreviation protection for PubMed text
# ============================================================================

# Medical abbreviations whose trailing period should NOT split a sentence
_MEDICAL_ABBREVS = [
    # Units & measurements
    'approx', 'avg', 'ca', 'dept', 'diam', 'div', 'est', 'fig', 'figs',
    'gov', 'hr', 'hrs', 'mg', 'ml', 'mo', 'mos', 'no', 'nos', 'pt', 'pts',
    'vol', 'wk', 'wks', 'yr', 'yrs', 'vs',
    # Medical/clinical
    'admin', 'adj', 'assoc', 'diff', 'diag', 'eval', 'max', 'min',
    'neg', 'pos', 'prev', 'prog', 'qual', 'quant', 'resp', 'sig',
    'supp', 'surg', 'ther', 'tx', 'dx',
    # Titles
    'dr', 'prof', 'mr', 'mrs', 'ms',
    # Latin / academic
    'al', 'ed', 'eds', 'eg', 'esp', 'ie', 'inc', 'ltd', 'corp', 'univ',
    # Genomic
    'del', 'ins', 'mut', 'amp', 'chr',
]

_PH = '\x00'  # null byte placeholder (never appears in real text)

def _protect_abbrevs(text: str) -> str:
    """Replace periods in known medical abbreviations with a placeholder."""
    # Multi-char abbreviations with internal periods (i.v., e.g., U.S., etc.)
    for pattern in [r'i\.v\.', r'i\.m\.', r's\.c\.', r'p\.o\.', r'e\.g\.', r'i\.e\.', r'U\.S\.', r'E\.U\.']:
        text = re.sub(pattern, lambda m: m.group().replace('.', _PH), text, flags=re.IGNORECASE)
    # et al.
    text = re.sub(r'\bet al\.', lambda m: m.group().replace('.', _PH), text)
    # Single-word abbreviations: preserve original case, replace only the trailing dot
    for abbr in _MEDICAL_ABBREVS:
        text = re.sub(
            r'\b(' + re.escape(abbr) + r')\.',
            lambda m: m.group(1) + _PH,
            text,
            flags=re.IGNORECASE,
        )
    return text

def _unprotect(text: str) -> str:
    return text.replace(_PH, '.')

_sent_segmenter = pysbd.Segmenter(language='en', clean=False)

def medical_sent_tokenize(text: str) -> list[str]:
    """Split text into sentences using pySBD with medical abbreviation protection.
    
    Handles: et al., i.v., vs., mos., pts., yrs., Dr., Fig., approx., del., etc.
    Returns a list of clean, non-empty sentences.
    """
    protected = _protect_abbrevs(text)
    raw_sents = _sent_segmenter.segment(protected)
    return [_unprotect(s).strip() for s in raw_sents if _unprotect(s).strip()]


# ============================================================================
# SENTENCE-AWARE CHUNKING
# ============================================================================

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap_sentences: int = OVERLAP_SENTENCES) -> list[str]:
    """Split text into chunks of complete sentences, with sentence-level overlap.
    
    Strategy:
      1. Tokenize text into sentences (medical-aware)
      2. Accumulate sentences until adding another would exceed chunk_size
      3. Emit current chunk; carry the last `overlap_sentences` sentences
         to the next chunk as context
      4. Every chunk starts and ends on a sentence boundary — never cuts
         mid-word or mid-thought
    
    Args:
        text: Input text (typically a PubMed passage = title + abstract)
        chunk_size: Soft character limit per chunk (may be slightly exceeded
                    if a single sentence is longer than chunk_size)
        overlap_sentences: Number of trailing sentences from the previous chunk
                          to include at the start of the next chunk
    
    Returns:
        List of text chunks, each containing complete sentences.
    """
    sentences = medical_sent_tokenize(text)
    if not sentences:
        return []

    chunks = []
    current_sents: list[str] = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent) + (1 if current_sents else 0)  # +1 for space join

        # If adding this sentence would exceed the limit and we already have content
        if current_len + sent_len > chunk_size and current_sents:
            # Emit current chunk
            chunk = ' '.join(current_sents)
            if len(chunk) > 50:
                chunks.append(chunk)

            # Carry overlap: last N sentences become the start of the next chunk
            if overlap_sentences > 0:
                carry = current_sents[-overlap_sentences:]
                current_sents = carry
                current_len = sum(len(s) for s in current_sents) + max(0, len(current_sents) - 1)
            else:
                current_sents = []
                current_len = 0

        current_sents.append(sent)
        current_len += sent_len

    # Emit remaining sentences
    if current_sents:
        chunk = ' '.join(current_sents)
        if len(chunk) > 50:
            chunks.append(chunk)

    return chunks


# ── Discover and preview source data ──────────────────────────────────
source_files = sorted(glob.glob(f"{SOURCE_CLEAN_DIR}/pubmed_*.jsonl"))

if not source_files:
    raise FileNotFoundError(
        f"No cleaned data found in {SOURCE_CLEAN_DIR}/\n"
        f"Run the cleaning script first:\n"
        f"  python {PROJECT_ROOT}/scripts/clean_pubmed.py"
    )

ORIGINAL_SOURCE_CLEAN_DIR = SOURCE_CLEAN_DIR

# ── Random sampling (if MAX_RECORDS > 0) ────────────────────
if MAX_RECORDS and MAX_RECORDS > 0:
    import random
    sampled_dir = f"{SOURCE_CLEAN_DIR}/sampled_{MAX_RECORDS}"

    # Reuse existing sampled files if they exist (preserves consistency with checkpoints)
    existing_sampled = sorted(glob.glob(f"{sampled_dir}/pubmed_*.jsonl")) if os.path.isdir(sampled_dir) else []
    if existing_sampled:
        source_files = existing_sampled
        SOURCE_CLEAN_DIR = sampled_dir
        total_available = sum(1 for sf in source_files for _ in open(sf))
        print(f"⚡ REUSING existing sample: {sampled_dir}/")
        print(f"  {total_available:,} records across {len(source_files)} files")
    else:
        # Load all records across all cancer types, then sample
        all_records = {}  # cancer_type -> list of records
        total_available = 0
        for filepath in source_files:
            ct = Path(filepath).stem
            with open(filepath) as f:
                records = [line for line in f]
            all_records[ct] = records
            total_available += len(records)
        
        if total_available > MAX_RECORDS:
            # Proportional sampling: each type keeps its share
            ratio = MAX_RECORDS / total_available
            os.makedirs(sampled_dir, exist_ok=True)
            new_source_files = []
            for ct, records in all_records.items():
                n = max(1, int(len(records) * ratio))
                sampled = random.sample(records, n)
                out_path = f"{sampled_dir}/{ct}.jsonl"
                with open(out_path, "w") as f:
                    f.writelines(sampled)
                new_source_files.append(out_path)
            source_files = sorted(new_source_files)
            SOURCE_CLEAN_DIR = sampled_dir  # redirect downstream lookups
            print(f"⚡ SAMPLED {MAX_RECORDS:,} records from {total_available:,} (ratio {ratio:.2%})")
            print(f"  Sampled files in: {sampled_dir}/")
        else:
            print(f"  Corpus has {total_available:,} records — no sampling needed (MAX_RECORDS={MAX_RECORDS:,})")
        del all_records

print(f"Found {len(source_files)} cancer type files\n")

total_records = 0
total_chunks = 0
cancer_type_stats = {}

for filepath in source_files:
    cancer_type = Path(filepath).stem  # e.g. "pubmed_bone_cancer"
    display_name = CANCER_TYPES.get(cancer_type, cancer_type)

    # Count records and estimate chunks without holding all in memory
    record_count = 0
    chunk_count = 0
    total_chars = 0

    with open(filepath) as f:
        for line in f:
            record = json.loads(line)
            passage = record.get("passage", "")
            record_count += 1
            total_chars += len(passage)
            chunk_count += len(chunk_text(passage))

    cancer_type_stats[cancer_type] = {
        "records": record_count,
        "chunks": chunk_count,
        "avg_chars": total_chars // max(record_count, 1),
    }
    total_records += record_count
    total_chunks += chunk_count

    print(f"  {display_name:25s} {record_count:>6,} records  {chunk_count:>6,} chunks  (avg {total_chars // max(record_count, 1):,} chars)")

print(f"\nTotal: {total_records:,} records → {total_chunks:,} chunks from {len(source_files)} cancer types")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"Estimated output: ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

# ── Check CancerGUIDE data ────────────────────────────────────────────
guide_files = sorted(glob.glob(f"{ORIGINAL_SOURCE_CLEAN_DIR}/cancerguide_*.jsonl"))
if guide_files:
    print(f"\nCancerGUIDE files found:")
    for gf in guide_files:
        count = sum(1 for _ in open(gf))
        print(f"  {Path(gf).name:40s} {count:>4} records")
else:
    print(f"\n⚠ No CancerGUIDE files found — treatment reasoning section will be skipped")

⚡ REUSING existing sample: /workspace/training/pubmed/data/source-clean/sampled_2/
  10 records across 10 files
Found 10 cancer type files

  Bone Cancer                    1 records       1 chunks  (avg 414 chars)
  Brain Tumour                   1 records       2 chunks  (avg 1,643 chars)
  Breast Cancer                  1 records       1 chunks  (avg 1,197 chars)
  Colon Cancer                   1 records       1 chunks  (avg 590 chars)
  Gastric Cancer                 1 records       1 chunks  (avg 419 chars)
  Kidney Cancer                  1 records       1 chunks  (avg 581 chars)
  Lung Cancer                    1 records       1 chunks  (avg 581 chars)
  Ovarian Cancer                 1 records       2 chunks  (avg 1,519 chars)
  Prostate Cancer                1 records       1 chunks  (avg 704 chars)
  Skin Cancer                    1 records       2 chunks  (avg 1,634 chars)

Total: 10 records → 13 chunks from 10 cancer types
Estimated output: ~39 QA pairs → ~9 conversations


## 4. Oncologist Persona

Single clinical oncologist persona with evidence-based reasoning style.
The system prompt establishes clinical competence, reasoning structure, and appropriate hedging.

In [4]:
# ============================================================================
# ONCOLOGIST PERSONA — Evidence-based clinical reasoning
# ============================================================================
# Unlike biblical (26 unique voices), we have ONE expert persona.
# Voice differentiation comes from cancer-type-specific knowledge, not identity.
# ============================================================================

ONCOLOGIST_SYSTEM_PROMPT = """You are a clinical oncologist with deep expertise in cancer biology, diagnosis, treatment planning, and patient care. You have extensive experience across medical oncology, surgical oncology, and radiation oncology.

YOUR REASONING APPROACH:
- Always think through the biological mechanism first, then clinical implications
- Cite specific pathways, genes, biomarkers, and staging systems when relevant
- Consider differential diagnoses and alternative interpretations
- Weigh evidence quality: randomized trials > cohort studies > case reports > expert opinion
- Acknowledge uncertainty explicitly — say "the evidence suggests" not "this proves"
- Consider patient-specific factors: comorbidities, performance status, prior treatments
- When discussing treatment, address efficacy, toxicity, and quality of life

YOUR COMMUNICATION STYLE:
- Structured and methodical — mechanism → evidence → clinical application → limitations
- Use precise medical terminology but explain complex concepts when teaching
- Reference specific studies, trials, or guidelines when applicable (e.g., NCCN, ASCO, ESMO)
- Distinguish between established standard-of-care and emerging/experimental approaches
- Always note when information may be evolving or when guidelines may have updated

RULES:
- Never fabricate specific statistics, trial names, or patient outcomes
- When unsure, state the limits of your knowledge explicitly
- Frame all clinical guidance with appropriate caveats about individualized care
- Do not provide direct patient-specific medical advice — frame as educational discussion
- Maintain scientific rigor while being accessible"""


def make_system_prompt(cancer_type: str = None) -> str:
    """Build system prompt, optionally with cancer-type specialization."""
    prompt = ONCOLOGIST_SYSTEM_PROMPT
    if cancer_type:
        display = CANCER_TYPES.get(cancer_type, cancer_type.replace("pubmed_", "").replace("_", " ").title())
        prompt += f"\n\nYou are currently discussing topics related to {display}. Draw on your specialized knowledge of this cancer type's biology, staging, standard treatments, and recent advances."
    return prompt


# Preview
print("ONCOLOGIST SYSTEM PROMPT")
print("=" * 60)
print(make_system_prompt("pubmed_lung_cancer")[:800])
print("...")
print(f"\nPrompt length: {len(make_system_prompt('pubmed_lung_cancer'))} chars")

ONCOLOGIST SYSTEM PROMPT
You are a clinical oncologist with deep expertise in cancer biology, diagnosis, treatment planning, and patient care. You have extensive experience across medical oncology, surgical oncology, and radiation oncology.

YOUR REASONING APPROACH:
- Always think through the biological mechanism first, then clinical implications
- Cite specific pathways, genes, biomarkers, and staging systems when relevant
- Consider differential diagnoses and alternative interpretations
- Weigh evidence quality: randomized trials > cohort studies > case reports > expert opinion
- Acknowledge uncertainty explicitly — say "the evidence suggests" not "this proves"
- Consider patient-specific factors: comorbidities, performance status, prior treatments
- When discussing treatment, address efficacy, toxicity, and qua
...

Prompt length: 1809 chars


## 5. DPO Preference Pair Collection

Collects preference pairs for **Direct Preference Optimization (DPO)** — the second training stage after SFT. DPO teaches the model to avoid fabricating specific data while still engaging substantively with questions.

**Three DPO pair sources:**

| Source | Chosen (Good) | Rejected (Bad) | Signal |
|--------|--------------|----------------|--------|
| **Grounding Rejects** | Knowledgeable answer using abstract + general medical knowledge | Answer that fabricates specific statistics/trials | Don't invent data — but DO use real knowledge |
| **Beyond-Evidence** | Knowledgeable answer acknowledging evidence gaps | Answer that fabricates specific data to fill gaps | Be transparent about evidence strength |
| **Self-Correction** | Corrected answer (from SFT artifacts) | Flawed answer (from SFT artifacts) | Fix mistakes when caught |

**Philosophy:** The "chosen" response should behave like a well-read oncologist — freely using
established medical knowledge while being transparent about what the specific abstract does/doesn't
cover. The "rejected" response fabricates specific data (fake trial names, invented statistics).
The model should learn: *use your knowledge freely, but never make up specifics*.

**Output format:** Chat-template DPO pairs compatible with TRL's DPOTrainer:
```json
{"chosen": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}],
 "rejected": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}],
 "source": "grounding_reject|beyond_evidence|self_correction"}
```

In [ ]:
# ============================================================================
# DPO PREFERENCE PAIR COLLECTION
# ============================================================================
# Collects (chosen, rejected) pairs from three sources for DPO fine-tuning.
# Each pair shares the same prompt (system + user) with different assistant
# responses, teaching the model to prefer grounded, honest, well-reasoned
# answers over hallucinated, evasive, or low-quality ones.
# ============================================================================

DPO_DIR = f"{OUTPUT_DIR}/dpo"
DPO_OUTPUT_FILE = f"{OUTPUT_ROOT}/pubmed_oncologist_v2_dpo.jsonl"
os.makedirs(DPO_DIR, exist_ok=True)

# ── Helper: build DPO pair in chat format ─────────────────────────────

def make_dpo_pair(system_prompt: str, user_msg: str, chosen_answer: str,
                  rejected_answer: str, source: str, cancer_type: str = "",
                  metadata: dict = None) -> dict:
    """Build a DPO pair in TRL chat format."""
    pair = {
        "chosen": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": chosen_answer},
        ],
        "rejected": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": rejected_answer},
        ],
        "source": source,
        "cancer_type": cancer_type,
    }
    if metadata:
        pair.update(metadata)
    return pair


# ======================================================================
# SOURCE 1: GROUNDING REJECTS — hallucinated answers vs re-generated
#            strict-grounded answers
# ======================================================================

STRICT_GROUNDING_PROMPT = """You are a knowledgeable clinical oncologist answering a question. A research abstract is provided as a reference, but you should also draw on your broader medical training and established oncology knowledge.

REFERENCE ABSTRACT:
---
{chunk}
---

Question: {question}

Rules:
- Use the abstract as your primary reference, but freely supplement with well-established medical knowledge (mechanisms, pathways, standard-of-care, widely-known trial results)
- When the abstract provides specific data, cite it; when you draw on general knowledge, that is fine — just be natural about it
- Do NOT fabricate specific statistics, p-values, patient counts, exact dosages, or trial names that you are not confident exist
- If information is uncertain or evolving, say so — distinguish established facts from emerging evidence
- Be thorough, clinically useful, and substantive"""

_tracker_dpo_grounding = None
_tracker_dpo_halluc = None


async def generate_strict_grounded_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate a strictly grounded answer for DPO 'chosen' example."""
    system_prompt = make_system_prompt(cancer_type)
    user_prompt = STRICT_GROUNDING_PROMPT.format(chunk=chunk, question=question)

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,  # Low temperature for strict grounding
                max_tokens=4096,
            ))
            if resp is None:
                _tracker_dpo_grounding.timeout()
                return ""
            answer = _extract_with_reasoning(resp)
            del resp
            _tracker_dpo_grounding.success()
            return process_answer(answer)
        except Exception as e:
            _tracker_dpo_grounding.error(e)
            return ""


print("COLLECTING DPO PREFERENCE PAIRS\n")
print(f"{'='*60}")
dpo_pairs_all = []

# ── Source 1: Grounding Rejects ──────────────────────────────────────
dpo_grounding_dir = f"{OUTPUT_DIR}/dpo/grounding_rejects"
grounding_reject_files = sorted(glob.glob(f"{dpo_grounding_dir}/*_rejects.jsonl"))
grounding_dpo_count = 0

if grounding_reject_files:
    print(f"\n[1/3] GROUNDING REJECTS — {len(grounding_reject_files)} type files")

    for reject_file in grounding_reject_files:
        cancer_type = Path(reject_file).stem.replace("_rejects", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        items = []
        with open(reject_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            continue

        # Rebuild chunk lookup for full chunks
        chunk_lookup = {}
        source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
        if os.path.exists(source_file):
            with open(source_file) as f:
                for line in f:
                    record = json.loads(line)
                    passage = record.get("passage", "")
                    for c in chunk_text(passage):
                        chunk_lookup[c[:100]] = c

        # Reset tracker
        _tracker_dpo_grounding = ApiErrorTracker(f"DPO-grounding/{display}")

        # Re-generate grounded answers for rejected questions
        regen_tasks = [
            generate_strict_grounded_answer(
                chunk_lookup.get(item.get("chunk_key", ""), ""),
                item["question"],
                cancer_type
            )
            for item in items
        ]
        chosen_answers = await atqdm.gather(*regen_tasks, desc=f"  {display} grounding DPO")
        _tracker_dpo_grounding.check_fatal()
        del regen_tasks, chunk_lookup

        report = _tracker_dpo_grounding.report()
        if report:
            print(report)

        # Build DPO pairs
        type_count = 0
        for item, chosen in zip(items, chosen_answers):
            if is_low_quality_answer(chosen):
                continue
            pair = make_dpo_pair(
                system_prompt=make_system_prompt(cancer_type),
                user_msg=item["question"],
                chosen_answer=chosen,
                rejected_answer=item["answer"],
                source="grounding_reject",
                cancer_type=cancer_type,
            )
            dpo_pairs_all.append(pair)
            type_count += 1

        grounding_dpo_count += type_count
        print(f"    {display:25s} {len(items):>4} rejects → {type_count:>4} DPO pairs")
        del items, chosen_answers
        gc.collect()
else:
    print("\n[1/3] GROUNDING REJECTS — no reject files found (run grounding check first)")

print(f"  Grounding DPO pairs: {grounding_dpo_count}")


# ======================================================================
# SOURCE 2: BEYOND-EVIDENCE — honest refusals vs hallucinated answers
# ======================================================================

HALLUCINATED_ANSWER_PROMPT = """You are a clinical oncologist who has been asked a question. IGNORE the fact that the evidence doesn't fully address this question — instead, provide a detailed, authoritative-sounding answer as if you have access to specific clinical data.

ABSTRACT FOR CONTEXT:
{chunk}

QUESTION: {question}

INSTRUCTIONS:
- Answer confidently as if the evidence fully supports your response
- INVENT specific details: exact percentages, study sizes, p-values, drug dosages
- Make up plausible-sounding trial names or cite invented guidelines
- The answer should sound authoritative and medically plausible
- Do NOT hedge or express uncertainty — present everything as established fact
- This is deliberately wrong for training purposes — do NOT add disclaimers

Provide the hallucinated answer directly."""


async def generate_hallucinated_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate a deliberately hallucinated answer for DPO 'rejected'."""
    prompt = HALLUCINATED_ANSWER_PROMPT.format(chunk=chunk, question=question)
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,  # High temperature for creative fabrication
                max_tokens=2048,
            ))
            if resp is None:
                _tracker_dpo_halluc.timeout()
                return ""
            # Use content only — no thinking for hallucinated examples
            answer = (resp.choices[0].message.content or "").strip()
            del resp
            _tracker_dpo_halluc.success()
            # Strip any thinking blocks
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
            return answer
        except Exception as e:
            _tracker_dpo_halluc.error(e)
            return ""


beyond_dir = f"{OUTPUT_DIR}/anti_hallucination/beyond_evidence"
beyond_files = sorted(glob.glob(f"{beyond_dir}/*_beyond.jsonl"))
beyond_dpo_count = 0

if beyond_files:
    print(f"\n[2/3] BEYOND-EVIDENCE — {len(beyond_files)} type files")

    for beyond_file in beyond_files:
        cancer_type = Path(beyond_file).stem.replace("_beyond", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        items = []
        with open(beyond_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            continue

        # Rebuild chunk lookup
        chunk_lookup = {}
        source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
        if os.path.exists(source_file):
            with open(source_file) as f:
                for line in f:
                    record = json.loads(line)
                    passage = record.get("passage", "")
                    for c in chunk_text(passage):
                        chunk_lookup[c[:100]] = c

        # Reset tracker
        _tracker_dpo_halluc = ApiErrorTracker(f"DPO-halluc/{display}")

        # Generate hallucinated answers as "rejected"
        halluc_tasks = [
            generate_hallucinated_answer(
                chunk_lookup.get(item.get("chunk_key", ""), ""),
                item["question"],
                cancer_type
            )
            for item in items
        ]
        halluc_answers = await atqdm.gather(*halluc_tasks, desc=f"  {display} beyond DPO")
        _tracker_dpo_halluc.check_fatal()
        del halluc_tasks
        chunk_lookup_saved = chunk_lookup  # Keep for chosen regeneration
        del chunk_lookup

        report = _tracker_dpo_halluc.report()
        if report:
            print(report)

        # Re-generate knowledgeable "chosen" answers (not rigid refusals)
        _tracker_dpo_beyond_chosen = ApiErrorTracker(f"DPO-beyond-chosen/{display}")

        chosen_tasks = [
            generate_strict_grounded_answer(
                chunk_lookup_saved.get(item.get("chunk_key", ""), ""),
                item["question"],
                cancer_type
            )
            for item in items
        ]
        beyond_chosen_answers = await atqdm.gather(*chosen_tasks, desc=f"  {display} beyond chosen")
        _tracker_dpo_beyond_chosen.check_fatal()
        del chosen_tasks

        report = _tracker_dpo_beyond_chosen.report()
        if report:
            print(report)

        # Build DPO pairs: chosen = knowledgeable answer, rejected = hallucinated
        type_count = 0
        for item, chosen, halluc in zip(items, beyond_chosen_answers, halluc_answers):
            if not halluc or len(halluc) < 50:
                continue
            if is_low_quality_answer(chosen):
                continue
            pair = make_dpo_pair(
                system_prompt=make_system_prompt(cancer_type),
                user_msg=item["question"],
                chosen_answer=chosen,
                rejected_answer=halluc,
                source="beyond_evidence",
                cancer_type=cancer_type,
            )
            dpo_pairs_all.append(pair)
            type_count += 1

        beyond_dpo_count += type_count
        print(f"    {display:25s} {len(items):>4} beyond → {type_count:>4} DPO pairs")
        del items, halluc_answers, beyond_chosen_answers, chunk_lookup_saved
        gc.collect()
else:
    print("\n[2/3] BEYOND-EVIDENCE — no beyond-evidence files found (run section 5d first)")

print(f"  Beyond-evidence DPO pairs: {beyond_dpo_count}")


# ======================================================================
# SOURCE 3: SELF-CORRECTION — corrected answers vs flawed answers
# ======================================================================

correction_dir = f"{OUTPUT_DIR}/anti_hallucination/self_correction"
correction_files = sorted(glob.glob(f"{correction_dir}/*_corrections.jsonl"))
correction_dpo_count = 0

if correction_files:
    print(f"\n[3/3] SELF-CORRECTION — {len(correction_files)} type files")

    for corr_file in correction_files:
        cancer_type = Path(corr_file).stem.replace("_corrections", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        type_count = 0
        with open(corr_file) as f:
            for line in f:
                conv = json.loads(line)
                turns = conv.get("conversations", [])

                # Expected format: system / human / gpt(flawed) / human(pushback) / gpt(corrected)
                if len(turns) < 5:
                    continue

                system_msg = turns[0]["value"] if turns[0]["from"] == "system" else ""
                question = turns[1]["value"] if turns[1]["from"] == "human" else ""
                flawed = turns[2]["value"] if turns[2]["from"] == "gpt" else ""
                corrected = turns[4]["value"] if turns[4]["from"] == "gpt" else ""

                if not question or not flawed or not corrected:
                    continue
                if len(corrected) < 50 or len(flawed) < 50:
                    continue

                pair = make_dpo_pair(
                    system_prompt=system_msg,
                    user_msg=question,
                    chosen_answer=corrected,
                    rejected_answer=flawed,
                    source="self_correction",
                    cancer_type=cancer_type,
                )
                dpo_pairs_all.append(pair)
                type_count += 1

        correction_dpo_count += type_count
        print(f"    {display:25s} → {type_count:>4} DPO pairs")
else:
    print("\n[3/3] SELF-CORRECTION — no correction files found (run section 5e first)")

print(f"  Self-correction DPO pairs: {correction_dpo_count}")


# ======================================================================
# NOTE: Source 4 (Quality Gate — borderline vs random good) was REMOVED.
# It paired borderline answers with random good answers from DIFFERENT
# questions, which is semantically wrong for DPO (the chosen/rejected
# must answer the SAME question). This would teach the model to prefer
# unrelated eloquent text over actual answers.
# ======================================================================


# ── Write all DPO pairs ──────────────────────────────────────────────
random.shuffle(dpo_pairs_all)

with open(DPO_OUTPUT_FILE, "w") as f:
    for pair in dpo_pairs_all:
        f.write(json.dumps(pair) + "\n")

print(f"\n{'='*60}")
print(f"DPO COLLECTION COMPLETE")
print(f"{'='*60}")
print(f"  Grounding rejects:  {grounding_dpo_count:>6,}")
print(f"  Beyond-evidence:    {beyond_dpo_count:>6,}")
print(f"  Self-correction:    {correction_dpo_count:>6,}")
print(f"  {'─'*30}")
print(f"  TOTAL DPO PAIRS:    {len(dpo_pairs_all):>6,}")
print(f"\n  Output: {DPO_OUTPUT_FILE}")
print(f"  Format: TRL chat-template DPO (chosen/rejected message lists)")

# Source distribution
from collections import Counter as DPOCounter
source_dist = DPOCounter(p["source"] for p in dpo_pairs_all)
print(f"\n  Source distribution:")
for src, cnt in source_dist.most_common():
    pct = cnt / max(len(dpo_pairs_all), 1) * 100
    print(f"    {src:<25} {cnt:>5} ({pct:.1f}%)")

del dpo_pairs_all
gc.collect()

COLLECTING DPO PREFERENCE PAIRS


[1/3] GROUNDING REJECTS — 10 type files


  Bone Cancer grounding DPO: 100%|██████████| 1963/1963 [49:31<00:00,  1.51s/it] 


    Bone Cancer               1963 rejects → 1960 DPO pairs


  Brain Tumour grounding DPO: 100%|██████████| 1470/1470 [32:31<00:00,  1.33s/it] 


    Brain Tumour              1470 rejects → 1469 DPO pairs


...

  Skin Cancer grounding DPO: 100%|██████████| 1483/1483 [30:43<00:00,  1.24s/it] 


    Skin Cancer               1483 rejects → 1477 DPO pairs
  Grounding DPO pairs: 14050

[2/3] BEYOND-EVIDENCE — 10 type files


  Bone Cancer beyond chosen: 100%|██████████| 1742/1742 [41:32<00:00,  1.43s/it] 


    Bone Cancer               1742 beyond → 1742 DPO pairs


  Brain Tumour beyond DPO:  28%|██▊       | 484/1727 [01:26<03:43,  5.57it/s]

## 10a. DPO Dataset Verification

Validates the DPO dataset: checks format consistency, length distributions, source balance, and samples from each source type.

In [ ]:
# ============================================================================
# DPO DATASET VERIFICATION
# ============================================================================

dpo_file = DPO_OUTPUT_FILE
if not os.path.exists(dpo_file):
    print(f"⚠ DPO file not found: {dpo_file}")
    print("  Run the DPO collection cell first.")
else:
    pairs = []
    with open(dpo_file) as f:
        for line in f:
            pairs.append(json.loads(line))

    print(f"DPO DATASET VERIFICATION")
    print(f"{'='*60}")
    print(f"  Total pairs:     {len(pairs):,}")
    print(f"  File:            {dpo_file}")
    print(f"  Size:            {os.path.getsize(dpo_file) / 1024 / 1024:.1f} MB")

    # Format checks
    format_errors = 0
    for i, p in enumerate(pairs):
        if "chosen" not in p or "rejected" not in p:
            format_errors += 1
            continue
        if not isinstance(p["chosen"], list) or not isinstance(p["rejected"], list):
            format_errors += 1
            continue
        if len(p["chosen"]) < 3 or len(p["rejected"]) < 3:
            format_errors += 1

    print(f"  Format errors:   {format_errors}")

    # Source distribution
    sources = Counter(p.get("source", "unknown") for p in pairs)
    print(f"\n  SOURCE DISTRIBUTION")
    print(f"  {'─'*40}")
    for src, cnt in sources.most_common():
        pct = cnt / len(pairs) * 100
        bar = "▪" * max(1, int(pct / 2))
        print(f"    {src:<25} {cnt:>5} ({pct:>5.1f}%) {bar}")

    # Cancer type coverage
    cancer_dist = Counter(p.get("cancer_type", "unknown") for p in pairs)
    print(f"\n  CANCER TYPE COVERAGE")
    print(f"  {'─'*40}")
    for ct, cnt in cancer_dist.most_common():
        display = CANCER_TYPES.get(ct, ct)
        print(f"    {display:<25} {cnt:>5}")

    # Length analysis
    chosen_lens = []
    rejected_lens = []
    for p in pairs:
        chosen_text = p["chosen"][-1]["content"] if p["chosen"] else ""
        rejected_text = p["rejected"][-1]["content"] if p["rejected"] else ""
        chosen_lens.append(len(chosen_text))
        rejected_lens.append(len(rejected_text))

    print(f"\n  LENGTH ANALYSIS")
    print(f"  {'─'*40}")
    print(f"    Chosen answers:   avg={sum(chosen_lens)//max(len(chosen_lens),1):,} chars "
          f"(min={min(chosen_lens, default=0):,}, max={max(chosen_lens, default=0):,})")
    print(f"    Rejected answers: avg={sum(rejected_lens)//max(len(rejected_lens),1):,} chars "
          f"(min={min(rejected_lens, default=0):,}, max={max(rejected_lens, default=0):,})")

    # Thinking block rates
    chosen_think = sum(1 for l in chosen_lens for p in [pairs[chosen_lens.index(l)]]
                       if has_thinking_block(p["chosen"][-1]["content"]))
    # Simpler approach:
    chosen_think = sum(1 for p in pairs if has_thinking_block(p["chosen"][-1].get("content", "")))
    rejected_think = sum(1 for p in pairs if has_thinking_block(p["rejected"][-1].get("content", "")))
    print(f"    Chosen with <think>:   {chosen_think}/{len(pairs)} ({chosen_think/max(len(pairs),1)*100:.1f}%)")
    print(f"    Rejected with <think>: {rejected_think}/{len(pairs)} ({rejected_think/max(len(pairs),1)*100:.1f}%)")

    # Samples
    print(f"\n  SAMPLE PAIRS (one per source)")
    print(f"  {'─'*40}")
    seen_sources = set()
    for p in pairs:
        src = p.get("source", "unknown")
        if src in seen_sources:
            continue
        seen_sources.add(src)

        user_msg = p["chosen"][1]["content"][:200] if len(p["chosen"]) > 1 else "N/A"
        chosen_preview = p["chosen"][-1]["content"][:300] if p["chosen"] else "N/A"
        rejected_preview = p["rejected"][-1]["content"][:300] if p["rejected"] else "N/A"

        print(f"\n    ── {src.upper()} ──")
        print(f"    [USER] {user_msg}...")
        print(f"    [CHOSEN] {chosen_preview}...")
        print(f"    [REJECTED] {rejected_preview}...")

    del pairs
    gc.collect()
    print(f"\n{'='*60}")
    print(f"✓ DPO verification complete")